### Análise Exploratória de Dados (EDA)

O objetivo desta etapa é explorar os dados, identificar padrões relevantes e responder às seguintes perguntas de negócio e analíticas:

- Qual é a distribuição da variável-alvo?
- Quais são as principais correlações e relações entre as variáveis ​​(features)?
- Quais testes estatísticos podem fornecer insights relevantes para a seleção de variáveis ​​e a modelagem?
- Qual é o impacto financeiro das fraudes nos diferentes segmentos de transação?
- Quais combinações de dispositivo e identidade são os indicadores mais fortes de fraude?
- Quais padrões de valor da transação diferenciam transações fraudulentas de legítimas?
- Quais variáveis ​​anônimas (V1–V339) possuem maior poder preditivo para a detecção de fraudes?
- Qual é o limiar de decisão ideal para a implementação em produção?

In [0]:
dbutils.library.restartPython()

In [0]:
dados = spark.sql("select * from fraud_detection_dev.silver.fraud_data_clean")

In [0]:
display(dados)

%md
### What is the distribution of the target variable?
------------------------------------------------------
### Qual é a distribuição da variável-alvo?

In [0]:
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. Target distribution - PySpark
# ============================================================

target_counts = (
    dados
    .groupBy("target")
    .agg(
        F.count("*").alias("count"),
        F.sum("TransactionAmt").alias("total_amount")
    )
)

total = dados.count()

target_dist = (
    target_counts
    .withColumn(
        "percentage",
        F.round((F.col("count") / F.lit(total)) * 100, 2)
    )
    .orderBy("target")
)

display(target_dist)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Apenas o resultado agregado
target_pd = target_dist.toPandas()

legitimate = target_pd.loc[target_pd["target"] == 0].iloc[0]
fraud = target_pd.loc[target_pd["target"] == 1].iloc[0]

sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(10, 3.2))

# ------------------------------------------------------------
# Composition bar
# ------------------------------------------------------------

ax.barh(
    y=0,
    width=legitimate["percentage"],
    left=0,
    height=0.32,
    color="#334155"
)

ax.barh(
    y=0,
    width=fraud["percentage"],
    left=legitimate["percentage"],
    height=0.32,
    color="#E5484D"
)

# ------------------------------------------------------------
# Labels
# ------------------------------------------------------------

# Legitimate
ax.text(
    legitimate["percentage"] / 2,
    0,
    f'{legitimate["percentage"]:.2f}%',
    ha="center",
    va="center",
    fontsize=13,
    color="white"
)

# Fraud
ax.text(
    legitimate["percentage"] - 1.5,
    0.32,
    f'{fraud["percentage"]:.2f}% Fraud',
    ha="center",
    va="bottom",
    fontsize=11,
    color="#E5484D"
)

# ------------------------------------------------------------
# Transaction counts
# ------------------------------------------------------------

ax.text(
    0,
    -0.35,
    f'{legitimate["count"]:,.0f} legitimate transactions',
    ha="left",
    va="center",
    fontsize=10,
    color="#64748B"
)

ax.text(
    100,
    -0.35,
    f'{fraud["count"]:,.0f} fraudulent transactions',
    ha="right",
    va="center",
    fontsize=10,
    color="#64748B"
)

# ------------------------------------------------------------
# Title
# ------------------------------------------------------------

ax.set_title(
    "Transaction Distribution",
    loc="left",
    fontsize=16,
    fontweight="normal",
    color="#0F172A",
    pad=28
)

ax.text(
    0,
    1.10,
    "Class distribution across the fraud detection dataset",
    transform=ax.transAxes,
    fontsize=10,
    color="#64748B"
)

# ------------------------------------------------------------
# Minimalist formatting
# ------------------------------------------------------------

ax.set_xlim(0, 100)
ax.set_ylim(-0.65, 0.65)

ax.set_xticks([])
ax.set_yticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

### Data Quality
-----------------
### Qualidade do dado

In [0]:
from pyspark.sql import functions as F

# Get total count
total_count = dados.count()

# Calculate quality metrics for all columns
quality_metrics = []

for col_name in dados.columns:
    # Base metrics for all columns
    col_stats = dados.agg(
        F.count(F.when(F.col(col_name).isNull(), 1)).alias("null_count"),
        F.countDistinct(col_name).alias("unique_count")
    ).collect()[0]
    
    null_count = col_stats["null_count"]
    unique_count = col_stats["unique_count"]
    
    metric = {
        "column": col_name,
        "null_count": null_count,
        "null_rate": round((null_count / total_count) * 100, 2),
        "unique_count": unique_count,
        "unique_rate": round((unique_count / total_count) * 100, 2)
    }
    
    quality_metrics.append(metric)

# Convert to DataFrame for display
quality_df = spark.createDataFrame(quality_metrics)

# Order by null_rate descending to see most problematic columns first
quality_df = quality_df.orderBy(F.col("null_rate").desc())

display(quality_df)

### Outliers detection
-----------------------
### Deteccao de outliers

**Isolation Forest**

For outlier detection, the unsupervised **Isolation Forest** algorithm will be used.

This method is suitable for the dataset due to its large number of records and multiple features. Unlike univariate statistical methods such as IQR or Z-Score, Isolation Forest can identify anomalous observations by considering the combined behavior of multiple features.

The algorithm is based on the principle that anomalous observations are rarer and more distinct from the majority of the data and, therefore, tend to be **isolated with fewer splits** in randomly generated trees.

In this project, the goal is not to assume that every outlier represents fraud. Instead, outlier detection will be used as an exploratory technique to identify transactions with unusual behavior and evaluate their relationship with the `target` variable.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, DoubleType, IntegerType
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest

# ============================================================
# 1. Select numeric features
# ============================================================

exclude_cols = [
    "TransactionID",
    "target",
    "TransactionDT",
    "ingestion_timestamp",
    "dataset_type"
]

numeric_cols = [
    field.name
    for field in dados.schema.fields
    if field.dataType.typeName()
    in ["double", "float", "integer", "long", "decimal"]
    and field.name not in exclude_cols
]

# ============================================================
# 2. Fill missing values with median
# ============================================================

median_values = {}
for col in numeric_cols:
    median_val = dados.approxQuantile(col, [0.5], 0.01)[0]
    median_values[col] = median_val if median_val is not None else 0.0

dados_filled = dados
for col, median_val in median_values.items():
    dados_filled = dados_filled.fillna({col: median_val})

In [0]:
# ============================================================
# 3. Train Isolation Forest
# ============================================================

training_data = dados_filled.select(numeric_cols).toPandas()

if_model = IsolationForest(
    contamination=0.05,
    n_estimators=300,
    max_samples=256,
    random_state=42,
    n_jobs=-1
)

if_model.fit(training_data)

In [0]:
# ============================================================
# 4. Apply model using pandas UDF
# ============================================================

from pyspark.sql.functions import pandas_udf, PandasUDFType

schema = StructType([
    StructField("anomaly_score", DoubleType(), True),
    StructField("is_outlier", IntegerType(), True)
])

@pandas_udf(schema, PandasUDFType.SCALAR)
def predict_outliers(*cols):
    df = pd.DataFrame({col_name: col for col_name, col in zip(numeric_cols, cols)})
    scores = if_model.decision_function(df)
    predictions = if_model.predict(df)
    predictions = np.where(predictions == -1, 1, 0)
    return pd.DataFrame({"anomaly_score": -scores, "is_outlier": predictions})

dados_with_outliers = dados_filled.withColumn("outlier_results",predict_outliers(*numeric_cols))

dados_with_outliers_final = (
    dados_with_outliers
    .withColumn("anomaly_score", F.col("outlier_results.anomaly_score"))
    .withColumn("is_outlier", F.col("outlier_results.is_outlier"))
    .drop("outlier_results")
)

In [0]:
# ============================================================
# 5. Outlier summary
# ============================================================

total = dados_with_outliers_final.count()

outlier_summary = (
    dados_with_outliers_final
    .groupBy("is_outlier")
    .agg(F.count("*").alias("count"))
    .withColumn("percentage", F.round(F.col("count") / F.lit(total) * 100, 2))
    .orderBy("is_outlier")
)

display(outlier_summary)


### Do the anomalous observations identified by Isolation Forest exhibit a higher concentration of fraud than the general population?
---------------------------------------------------------------------------------------------------------------------
### As observações consideradas anômalas pelo Isolation Forest apresentam uma concentração de fraude maior que a população geral?

In [0]:
# ============================================================
# 6. Relationship between anomalies and fraud
# ============================================================

outlier_fraud_analysis = (
    dados_with_outliers_final
    .groupBy("is_outlier")
    .agg(
        F.count("*").alias("transactions"),
        F.sum("target").alias("fraud_transactions"),
        F.round(F.avg("target") * 100, 2).alias("fraud_rate_pct"),
        F.round(F.avg("anomaly_score"), 4).alias("avg_anomaly_score")
    )
)

display(outlier_fraud_analysis)

# ============================================================
# 7. Update main DataFrame
# ============================================================

dados = dados_with_outliers_final

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Prepare data
# ============================================================

outlier_pd = outlier_fraud_analysis.toPandas()

outlier_pd["group"] = outlier_pd["is_outlier"].map({
    1: "Outliers",
    0: "Regular transactions"
})

# Ordenação
outlier_pd = outlier_pd.sort_values(
    "fraud_rate_pct",
    ascending=True
)

# ============================================================
# Executive visualization
# ============================================================

sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(9, 4))

colors = [
    "#475569" if group == "Regular transactions" else "#E5484D"
    for group in outlier_pd["group"]
]

bars = ax.barh(
    outlier_pd["group"],
    outlier_pd["fraud_rate_pct"],
    height=0.42,
    color=colors
)

# ============================================================
# Labels
# ============================================================

for bar, (_, row) in zip(bars, outlier_pd.iterrows()):

    width = bar.get_width()
    y = bar.get_y() + bar.get_height() / 2

    # Main metric
    ax.text(
        width + 0.35,
        y + 0.06,
        f"{row['fraud_rate_pct']:.2f}%",
        va="center",
        ha="left",
        fontsize=14,
        color="#0F172A"
    )

    # Secondary information
    ax.text(
        width + 0.35,
        y - 0.10,
        f"{row['fraud_transactions']:,.0f} frauds  •  "
        f"{row['transactions']:,.0f} transactions",
        va="center",
        ha="left",
        fontsize=9,
        color="#64748B"
    )

# ============================================================
# Title and subtitle
# ============================================================

ax.set_title(
    "Fraud Rate by Anomaly Classification",
    loc="left",
    fontsize=16,
    fontweight="normal",
    color="#0F172A",
    pad=28
)

ax.text(
    0,
    1.05,
    "Fraud incidence is substantially higher among anomalous transactions",
    transform=ax.transAxes,
    fontsize=10,
    color="#64748B"
)

# ============================================================
# Formatting
# ============================================================

ax.set_xlabel("")
ax.set_ylabel("")

ax.set_xlim(
    0,
    outlier_pd["fraud_rate_pct"].max() * 1.55
)

ax.grid(False)

ax.tick_params(
    axis="y",
    length=0,
    labelsize=10,
    colors="#334155"
)

ax.set_xticks([])

for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

### Análise

Entre as transações identificadas como anômalas, **17,45% são fraudulentas**. Considerando a quantidade de registros e a natureza do problema, não será realizada a remoção desses valores, pois eles podem representar comportamentos naturais do fenômeno analisado.

Este estudo possui caráter exploratório, com o objetivo de compreender a relação entre valores discrepantes ou comportamentos anômalos nos dados e a ocorrência de transações fraudulentas. A remoção dessas observações poderia eliminar padrões relevantes para a identificação de fraudes.

------------------------------------------------------

### Analysis

Among the transactions identified as anomalous, **17.45% are fraudulent**. Considering the number of records and the nature of the problem, these observations will not be removed, as they may represent natural behaviors of the phenomenon being analyzed.

This analysis has an exploratory purpose, aiming to understand the relationship between outliers or anomalous patterns in the data and the occurrence of fraudulent transactions. Removing these observations could eliminate relevant patterns for fraud detection.

### What are the main correlations and relationships between features?
### Quais são as principais correlações e relações entre as variáveis (features)?

In [0]:
# remove vars
remove = ['ingestion_timestamp', 'dataset_type']
dados = dados.drop(*remove)

In [0]:
display(dados.limit(10))

In [0]:
from pyspark.ml.feature import StringIndexer
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# ============================================================
# 1. Identify categorical columns
# ============================================================

categorical_cols = [
    field.name
    for field in dados.schema.fields
    if field.dataType.typeName() == "string"
]

# ============================================================
# 2. Apply Label Encoding using StringIndexer
# ============================================================

dados_encoded = dados

for col in categorical_cols:
    indexer = StringIndexer(
        inputCol=col,
        outputCol=f"{col}_indexed",
        handleInvalid="keep"
    )
    indexer_model = indexer.fit(dados_encoded)
    dados_encoded = indexer_model.transform(dados_encoded)
    dados_encoded = dados_encoded.drop(col).withColumnRenamed(f"{col}_indexed", col)

# ============================================================
# 3. Select columns for correlation (exclude anomaly_score)
# ============================================================

exclude_corr = ["TransactionID", "TransactionDT", "anomaly_score", "is_outlier"]

corr_cols = [
    field.name
    for field in dados_encoded.schema.fields
    if field.dataType.typeName() in ["double", "float", "integer", "long", "decimal"]
    and field.name not in exclude_corr
]

# ============================================================
# 4. Calculate correlation matrix
# ============================================================

corr_data = dados_encoded.select(corr_cols).toPandas()
corr_matrix = corr_data.corr()

# ============================================================
# 5. Filter high correlations with target
# ============================================================

target_corr = corr_matrix["target"].drop("target").abs().sort_values(ascending=False)
top_features = target_corr.head(40)

display(top_features.to_frame(name="correlation_with_target"))

# ============================================================
# 6. Visualize correlation matrix
# ============================================================

top_feature_names = top_features.index.tolist() + ["target"]
corr_subset = corr_matrix.loc[top_feature_names, top_feature_names]

sns.set_theme(style="white")

fig, ax = plt.subplots(figsize=(14, 12))

mask = np.triu(np.ones_like(corr_subset, dtype=bool))

sns.heatmap(
    corr_subset,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8},
    ax=ax,
    vmin=-1,
    vmax=1
)

ax.set_title(
    "Correlation Matrix - Top 20 Features vs Target",
    fontsize=16,
    fontweight="normal",
    pad=20
)

plt.tight_layout()
plt.show()

**Analise**

Muitas variaveis possuem multicolinearidade, nessa quantidade, ate mesmos modelos de arvore terao retrabalho em computar todas elas.

### What statistical tests can provide relevant insights for feature selection and modeling?
### Quais testes estatísticos podem fornecer insights relevantes para a seleção de variáveis e a modelagem?

**Teste Chi-Quadrado de Idependencia**

Metodo estatistico utilizado para verificar se existe associacao entre duas variaveis categoricas.

- Hipotese nula, A variavel explicativa e o target sao independentes
- Hipotese alternativa - Existe associacao entre a variavel explicativa e o target

O teste compara as frequencias observadas com as frequencias esperadas caso nao existisse qualquer associacao entre as variaveis

**V de Cramer**

Teste estatistico que quantifica a intensidade da associacao entre duas variaveis categoricas. A diferenca entre ele e o teste Qui-Quadrado eh que o V de Cramer realiza uma normalizacao da estatistica, permitindo comparar variaveis mesmo quando a base possui um grande numero de observacoes

A normalizacao torna o resultado independente da escala da amostra, fazendo com que o V de Cramer assuma sempre valores entre 0 e 1.

**Ks Univariado**

Adaptacao do teste de kolmogorov-Smirnov para avaliar uma unica variavel por vez. Esse eh um teste estatistico nao parametrico ou sejam nao existe a hipotese de que uma determinada variavel possui algum comportamento de uma distribuicao teorica. Ele vai responder a seguinte pergunta, a distribuicao dessa variavel eh diferente entre os clientes adimplentes e inadimplentes?


In [0]:
from pyspark.sql import functions as F
from scipy.stats import ks_2samp
import pandas as pd

# ============================================================
# 1. Select continuous variables
# ============================================================

exclude_ks = ["TransactionID", "TransactionDT", "target", "anomaly_score", "is_outlier"]

continuous_cols = [
    field.name
    for field in dados.schema.fields
    if field.dataType.typeName() in ["double", "float"]
    and field.name not in exclude_ks
]

# ============================================================
# 2. Calculate KS statistic for each variable
# ============================================================

ks_results = []

for col in continuous_cols:
    # Get data for each target group
    data_0 = dados.filter(F.col("target") == 0).select(col).dropna().toPandas()[col].values
    data_1 = dados.filter(F.col("target") == 1).select(col).dropna().toPandas()[col].values
    
    if len(data_0) > 0 and len(data_1) > 0:
        ks_stat, p_value = ks_2samp(data_0, data_1)
        
        ks_results.append({
            "variable": col,
            "ks_statistic": float(round(ks_stat, 4)),
            "p_value": float(round(p_value, 6))
        })

# ============================================================
# 3. Create DataFrame with results
# ============================================================

statistical_tests_df = spark.createDataFrame(ks_results)

# Add interpretation based on KS value
statistical_tests_df = statistical_tests_df.withColumn(
    "interpretation",
    F.when(F.col("ks_statistic") > 0.3, "High discriminatory power")
     .when(F.col("ks_statistic") > 0.1, "Moderate discriminatory power")
     .otherwise("Low discriminatory power")
)

statistical_tests_df = statistical_tests_df.orderBy(F.col("ks_statistic").desc())

# ============================================================
# 4. Display top 6 variables with highest KS
# ============================================================

top_ks_vars = statistical_tests_df.limit(6)

display(top_ks_vars)

In [0]:
from pyspark.sql import functions as F
from scipy.stats import chi2_contingency
import pandas as pd
import numpy as np

# ============================================================
# 1. Select categorical variables
# ============================================================

exclude_cat = ["TransactionID", "target"]

categorical_cols = [
    field.name
    for field in dados.schema.fields
    if field.dataType.typeName() in ["string", "integer", "long"]
    and field.name not in exclude_cat
    and not field.name.startswith("V")  # Exclude anonymous V variables
    and field.name not in ["TransactionDT", "anomaly_score", "is_outlier"]
]

# ============================================================
# 2. Calculate Chi-Square and Cramer's V
# ============================================================

chi_results = []

for col in categorical_cols:
    # Create contingency table
    contingency_spark = (
        dados
        .groupBy(col, "target")
        .count()
        .groupBy(col)
        .pivot("target")
        .sum("count")
        .fillna(0)
    )
    
    contingency_pd = contingency_spark.toPandas()
    
    if contingency_pd.shape[0] > 1 and len(contingency_pd.columns) > 1:
        # Remove the grouping column to get only counts
        contingency_table = contingency_pd.iloc[:, 1:].values
        
        # Calculate Chi-Square
        chi2, p_value, dof, expected = chi2_contingency(contingency_table)
        
        # Calculate Cramer's V
        n = contingency_table.sum()
        min_dim = min(contingency_table.shape[0] - 1, contingency_table.shape[1] - 1)
        cramers_v = np.sqrt(chi2 / (n * min_dim)) if min_dim > 0 else 0
        
        chi_results.append({
            "variable": col,
            "chi2_statistic": float(round(chi2, 4)),
            "cramers_v": float(round(cramers_v, 4)),
            "p_value": float(round(p_value, 6))
        })

# ============================================================
# 3. Create DataFrame and add interpretation
# ============================================================

chi_tests_df = spark.createDataFrame(chi_results)

# Add interpretation based on Cramer's V value
chi_tests_df = chi_tests_df.withColumn(
    "interpretation",
    F.when(F.col("cramers_v") > 0.3, "Strong association")
     .when(F.col("cramers_v") > 0.1, "Moderate association")
     .otherwise("Weak association")
)

chi_tests_df = chi_tests_df.orderBy(F.col("cramers_v").desc())

display(chi_tests_df)

# ============================================================
# 4. Add test_type column and union with KS results
# ============================================================

ks_with_type = statistical_tests_df.withColumn("test_type", F.lit("KS"))

chi_with_type = (
    chi_tests_df
    .withColumn("test_type", F.lit("Chi-Square"))
    .withColumn("ks_statistic", F.lit(None).cast("double"))
    .select("variable", "test_type", "ks_statistic", "chi2_statistic", "cramers_v", "p_value", "interpretation")
)

ks_with_cols = (
    ks_with_type
    .withColumn("chi2_statistic", F.lit(None).cast("double"))
    .withColumn("cramers_v", F.lit(None).cast("double"))
    .select("variable", "test_type", "ks_statistic", "chi2_statistic", "cramers_v", "p_value", "interpretation")
)

statistical_tests_df = ks_with_cols.union(chi_with_type)

### What is the financial impact of fraud across different transaction segments?
### Qual é o impacto financeiro das fraudes nos diferentes segmentos de transação?

In [0]:
display(dados)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F


def format_currency(value):
    if value >= 1_000_000:
        return f"${value / 1_000_000:.2f}M"
    elif value >= 1_000:
        return f"${value / 1_000:.1f}K"
    return f"${value:,.0f}"


def plot_fraud_amount_by_segment(
    df,
    segment,
    top_n=10
):

    fraud_segment = (
        df
        .filter(F.col("target") == 1)
        .groupBy(segment)
        .agg(
            F.sum("TransactionAmt").alias("fraud_amount"),
            F.count("*").alias("fraud_transactions")
        )
        .filter(F.col(segment).isNotNull())
        .orderBy(F.col("fraud_amount").desc())
        .limit(top_n)
        .toPandas()
    )

    fraud_segment = fraud_segment.sort_values(
        "fraud_amount",
        ascending=True
    )

    sns.set_theme(style="white")

    fig, ax = plt.subplots(figsize=(10, 5.5))

    bars = ax.barh(
        fraud_segment[segment].astype(str),
        fraud_segment["fraud_amount"],
        height=0.48,
        color="#D9534F"
    )

    max_value = fraud_segment["fraud_amount"].max()

    # Espaço fixo para os labels
    label_x = max_value * 1.04

    for bar, (_, row) in zip(
        bars,
        fraud_segment.iterrows()
    ):

        ax.text(
            label_x,
            bar.get_y() + bar.get_height() / 2,
            format_currency(row["fraud_amount"]),
            va="center",
            ha="left",
            fontsize=10,
            color="#334155"
        )

    # Title
    ax.set_title(
        f"Fraudulent Transaction Value by {segment}",
        loc="left",
        fontsize=15,
        fontweight="normal",
        color="#0F172A",
        pad=24
    )

    ax.text(
        0,
        1.02,
        f"Top {top_n} categories ranked by fraudulent transaction value",
        transform=ax.transAxes,
        fontsize=9.5,
        color="#64748B"
    )

    # Formatting
    ax.set_xlabel("")
    ax.set_ylabel("")

    ax.set_xticks([])

    ax.tick_params(
        axis="y",
        length=0,
        labelsize=10,
        colors="#334155",
        pad=8
    )

    ax.set_xlim(
        0,
        max_value * 1.18
    )

    ax.grid(False)

    for spine in ax.spines.values():
        spine.set_visible(False)

    plt.tight_layout()
    plt.show()


for segment in segments:
    plot_fraud_amount_by_segment(
        dados,
        segment,
        top_n=10
    )

### Which device and identity combinations are the strongest fraud indicators?
### Quais combinações de dispositivo e identidade são os indicadores mais fortes de fraude?

### What transaction amount patterns differentiate fraudulent from legitimate transactions?
### Quais padrões de valor da transação diferenciam transações fraudulentas de legítimas?

### Which anonymous variables (V1–V339) have the greatest predictive power for fraud detection?
### Quais variáveis anônimas (V1–V339) possuem maior poder preditivo para a detecção de fraudes?

### Feature Engineer